# Phase Difference Analysis: Directional vs Ambient Sound Discrimination

**Goal:** Validate the hypothesis that ambient/diffuse sounds have low or spatially-predictable inter-mic phase differences,
while true directional sources produce consistent, frequency-linear phase slopes across mic pairs (TDOA-driven).

**Why this matters for ODAS:**
- Current FP root cause: array geometry hotspots (`-90°, 0°, -120°`) where GCC-PHAT correlation peaks arise from *ambient noise*, not real sources
- Current miss rate: confirmation window (`N_prob=6`) kills short/quiet events
- **Proposed fix:** A per-frame *directionality score* based on phase coherence, used as a pre-gate before GCC-PHAT peak-picking

**DSP intuition:**
- Directional source at azimuth θ → TDOA between mics is fixed → cross-spectrum phase = ω·TDOA (linear in frequency, consistent over time)
- Diffuse ambient field → cross-spectrum phase is random (or follows the sinc model for spatially incoherent noise)
- **Key discriminator:** phase slope R² + deviation from diffuse-field spatial coherence model

**Scene:** `forest_animals_20260409_103128` — 1220s, 285 GT directional events, real forest ambient at −10 dBFS

## 1. Import Required Libraries

In [1]:
import numpy as np
import json
import warnings
from pathlib import Path
from scipy import signal as scipy_signal
from scipy.stats import circstd, circmean
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.colors import Normalize
import matplotlib.cm as cm
import pandas as pd

warnings.filterwarnings('ignore')

# ── Mic array geometry (ReSpeaker USB 4-Mic Array, square, 64mm diameter) ──
MIC_POSITIONS = np.array([
    [-0.032,  0.000, 0.000],   # Mic 0: Left   (-X)
    [ 0.000, -0.032, 0.000],   # Mic 1: Back   (-Y)
    [ 0.032,  0.000, 0.000],   # Mic 2: Right  (+X)
    [ 0.000,  0.032, 0.000],   # Mic 3: Front  (+Y)
])
MIC_PAIRS = [(0,2), (1,3), (0,1), (1,2), (2,3), (3,0)]  # all 6 pairs
PAIR_LABELS = ['L-R', 'B-F', 'L-B', 'B-R', 'R-F', 'F-L']
SPEED_OF_SOUND = 343.0
SR = 16000

# ── Scene & audio paths ──
SCENE_FILE   = Path('/home/azureuser/config/scenes/forest animals.json')
RAW_AUDIO    = Path('/home/azureuser/simulator/outputs/renders/forest_animals_20260409_103128.raw')

# ── STFT parameters (match ODAS hop=128, frame=512 for better freq resolution) ──
FRAME_SIZE = 512          # samples  (~32ms)
HOP_SIZE   = 128          # samples  (~8ms — same as ODAS)
N_FREQS    = FRAME_SIZE // 2 + 1  # 257

print(f"Sample rate     : {SR} Hz")
print(f"Frame size      : {FRAME_SIZE} samples = {1000*FRAME_SIZE/SR:.1f} ms")
print(f"Hop size        : {HOP_SIZE} samples  = {1000*HOP_SIZE/SR:.1f} ms")
print(f"Frequency bins  : {N_FREQS}")
print(f"Max TDOA (diag) : {np.sqrt(2)*0.064/SPEED_OF_SOUND*1000:.3f} ms  "
      f"→ max unambiguous freq for 32mm pair: {SPEED_OF_SOUND/(2*0.064):.0f} Hz")

Sample rate     : 16000 Hz
Frame size      : 512 samples = 32.0 ms
Hop size        : 128 samples  = 8.0 ms
Frequency bins  : 257
Max TDOA (diag) : 0.264 ms  → max unambiguous freq for 32mm pair: 2680 Hz


## 2. Load and Parse Audio Scene Data

Load the 6-channel raw audio (channels 1–4 are the mics), and parse the scene JSON
to get ground-truth directional event windows (start_time, end_time, azimuth, label).

We build **three tiers** of segments:
- **Tier 1 — GT directional events** — known single-source windows (bear, elephant, drone, etc.)
- **Tier 2 — Real ambient windows** — gaps between GT events in the forest recording (contains *distributed* directional sources from the background, but no dominant isolated event)
- **Tier 3 — Synthetic isotropic noise** — mathematical diffuse field (sum of 120 random-azimuth plane waves) → MSC = sinc² exactly; this is the theoretical floor our discriminator must beat

### ⚠️ Fundamental Design Clarification: What the Pipeline Actually Looks Like

**The rendering pipeline is:**
```
ambient_raw.wav  ──┐
                   ├──► pyroomacoustics renderer ──► 6-channel output ──► ODAS
elephant.wav ──────┘   (spatial mix, reverb)
bear.wav   ─────────┘
drone.wav  ─────────┘
```

The **ambient raw recording is a real captured scene** — and it is full of its own directional sources:
- A bird sitting on the same branch as the mic → strong, persistent, directional at some azimuth
- Children playing 10m to the left → directional at ~−90°
- A stream 20m ahead → directional at ~0°  
- Distant traffic → directional at +45°

These are **not diffuse**. They will produce sharp GCC-PHAT peaks, clear phase slopes, high MSC — exactly like the "target" elephant or bear we overlaid.

**This is the actual hard problem:**

| Source type | Phase coherent? | Loud? | We want it detected? |
|---|---|---|---|
| Elephant (overlaid, GT event) | ✅ Yes | Varies | ✅ Yes |
| Bird on nearby branch (from ambient raw) | ✅ Yes | Often loud | ❌ No (unless we're detecting birds) |
| Children playing (from ambient raw) | ✅ Yes | Loud | ❌ No |
| Pure wind / HVAC / far traffic | Partially | Can be loud | ❌ No |
| Array geometry hotspot artifact | Partially (fake peak) | — | ❌ No |

**Phase coherence alone cannot tell the bird from the elephant.** They both look directional to the mic array.

### What phase coherence CAN and CANNOT do

**CAN suppress:**
- Purely diffuse components (the "acoustic bath" — isotropic reverb, rain, white noise)
- Array geometry hotspot artifacts at −90°/0°/−120° that arise from the *averaging* of many incoherent sources perfectly aligned to inter-mic axes
- Frames where multiple competing sources partially cancel each other's coherence (very busy scenes)

**CANNOT suppress:**
- Any persistent real directional source from the ambient recording (bird, children, stream)
- These look identical to target events in phase space

### What this means for our experiment design

The three-tier labelling still holds, but with corrected interpretation:

| Tier | What it actually contains | MSC expected | Purpose in experiment |
|---|---|---|---|
| **GT event windows** | Ambient background + overlaid target (elephant etc.) | High (target dominates if loud enough) | Positive class |
| **Real ambient gaps** | Ambient background only — bird, children, wind, all the persistent directional sources | **Also high** — the bird is still there | This is the hard negative class — NOT diffuse |
| **Synthetic isotropic** | Mathematical diffuse field | sinc² exactly | Theoretical lower bound only |

**The gap between "GT event window" and "real ambient gap" MSC is the SNR of the overlaid source above the ambient background.** If the elephant is quiet or far away, that gap shrinks to zero and phase coherence is useless.

### Revised hypothesis

Phase coherence is **not a standalone event detector**. It is a **pre-filter for two specific cases**:
1. Kill frames that are genuinely near-diffuse (suppresses GCC-PHAT's structural hotspot artifacts)
2. Give a per-frame *excess coherence* signal that, **combined with YAMNet content classification**, narrows down candidate azimuths

The real separation between "bird" and "elephant" **must come from YAMNet** (content), not from phase geometry.

The experiment below still produces useful numbers: it tells us how much *excess* coherence a strong overlaid target adds above the ambient baseline. If that excess is measurable, we can use it to weight SSL peaks. If it isn't, phase gating is only useful for the diffuse/hotspot case.

In [ ]:
def generate_synthetic_diffuse(duration_s, n_sources=120, sr=SR, seed=42):
    """
    Generate a synthetic isotropic diffuse noise field for 4 mics.

    Method: sum N plane waves arriving from uniformly random azimuths, each
    carrying bandlimited noise. At large N this converges to the theoretical
    diffuse field whose inter-mic MSC = sinc²(2πfd/c).

    Returns mic_audio (4, n_samples) float32.
    """
    rng_s = np.random.default_rng(seed)
    n_samples = int(duration_s * sr)
    mic_audio_out = np.zeros((4, n_samples), dtype=np.float32)

    # Random azimuths uniformly distributed on the horizontal plane
    azimuths = rng_s.uniform(0, 2 * np.pi, n_sources)

    for az in azimuths:
        # Unit direction vector (horizontal plane)
        direction = np.array([np.cos(az), np.sin(az), 0.0])

        # Independent bandlimited noise source (pink-ish: random spectrum)
        noise = rng_s.standard_normal(n_samples).astype(np.float32)
        # Bandpass 200–5000 Hz to match typical animal/drone content
        sos = scipy_signal.butter(4, [200, 5000], btype='band', fs=sr, output='sos')
        noise = scipy_signal.sosfilt(sos, noise).astype(np.float32)
        noise /= (np.std(noise) + 1e-9)   # normalise per-source

        for mic_idx, pos in enumerate(MIC_POSITIONS):
            # Propagation delay for this mic and direction
            delay_s   = np.dot(pos, direction) / SPEED_OF_SOUND
            delay_smp = delay_s * sr            # fractional samples

            # Integer shift + linear interpolation for sub-sample accuracy
            d_int  = int(np.floor(delay_smp))
            d_frac = delay_smp - d_int
            shifted = np.roll(noise, d_int).astype(np.float64)
            if abs(d_int) > 0:
                # Zero out wrap-around region
                if d_int > 0:
                    shifted[:d_int] = 0
                else:
                    shifted[d_int:] = 0
            # Sub-sample interpolation
            if d_frac != 0 and len(shifted) > 1:
                shifted = (1 - d_frac) * shifted + d_frac * np.roll(shifted, 1)
            mic_audio_out[mic_idx] += shifted.astype(np.float32)

    # Normalise whole array to −20 dBFS (similar to real ambient level)
    rms = np.sqrt(np.mean(mic_audio_out**2))
    target_rms = 10**(-20/20)            # −20 dBFS
    mic_audio_out *= target_rms / (rms + 1e-9)
    return mic_audio_out


# Generate 3 seconds of synthetic diffuse noise
print("Generating synthetic isotropic diffuse noise (N=120 plane waves) …")
synth_diffuse = generate_synthetic_diffuse(duration_s=3.0, n_sources=120)
print(f"  shape: {synth_diffuse.shape}  RMS: {np.sqrt(np.mean(synth_diffuse**2)):.4f}")

# Verify it matches sinc model — compute MSC for L-R pair
stft_sd = stft_mics(synth_diffuse)
msc_sd = compute_msc_spectrum(stft_sd) if 'compute_msc_spectrum' in dir() else {}

# Quick sanity-check plot
fig, ax = plt.subplots(1, 1, figsize=(8, 4))
alias_bin_preview = int(SPEED_OF_SOUND / (2 * 0.064) / (SR / FRAME_SIZE))
d_lr = np.linalg.norm(MIC_POSITIONS[2] - MIC_POSITIONS[0])
diffuse_theory = np.sinc(2 * freqs[:alias_bin_preview] * d_lr / SPEED_OF_SOUND)**2  # sinc²

stft_check = stft_mics(synth_diffuse)
xi = stft_check[0]; xj = stft_check[2]
Pxy = np.mean(xi * np.conj(xj), axis=1)
Pxx = np.mean(np.abs(xi)**2, axis=1)
Pyy = np.mean(np.abs(xj)**2, axis=1)
msc_synth_lr = np.clip(np.abs(Pxy)**2 / (Pxx * Pyy + 1e-30), 0, 1)

ax.plot(freqs[:alias_bin_preview], msc_synth_lr[:alias_bin_preview],
        color='darkorange', lw=1.8, label='Synthetic diffuse (measured MSC)')
ax.plot(freqs[:alias_bin_preview], diffuse_theory,
        color='black', lw=1.5, ls='--', label='Theoretical sinc² model')
ax.set_xlabel('Frequency (Hz)')
ax.set_ylabel('MSC  (L-R pair)')
ax.set_title('Sanity check: Synthetic diffuse field vs sinc² model\n'
             '(good match → generator is correct)')
ax.legend()
ax.grid(True, alpha=0.3)
ax.set_ylim(0, 1.1)
plt.tight_layout()
plt.show()
print("If the orange line tracks the dashed black line → diffuse generator is correct.")

In [ ]:
# ── Load raw audio ──────────────────────────────────────────────────────────
print("Loading audio …")
raw = np.fromfile(RAW_AUDIO, dtype=np.int16)
n_samples = len(raw) // 6
audio_6ch = raw.reshape(n_samples, 6).T.astype(np.float32) / 32767.0
# Channels 2-5 (0-indexed 1-4) are the four mics
mic_audio = audio_6ch[1:5, :]   # shape (4, n_samples)
duration_s = n_samples / SR
print(f"  Loaded {n_samples:,} samples — {duration_s:.1f}s — 4 mic channels")

# ── Parse scene JSON → ground-truth event list ───────────────────────────────
scene = json.loads(SCENE_FILE.read_text())
directional = scene['directional_sources']

events = []
for src in directional:
    x, y = src['x'], src['y']
    azimuth_deg = np.degrees(np.arctan2(y, x))
    dist_m = np.sqrt(x**2 + y**2)
    events.append({
        'label'      : src['label'],
        'start'      : src['start_time'],
        'end'        : src['end_time'],
        'azimuth_deg': azimuth_deg,
        'dist_m'     : dist_m,
        'volume'     : src.get('volume', 1.0),
    })

events.sort(key=lambda e: e['start'])
df_events = pd.DataFrame(events)
print(f"\n  {len(df_events)} GT directional events")
print(f"  Labels: {df_events['label'].value_counts().to_dict()}")
print(f"\n  Duration range: {df_events['end'].max():.1f}s  "
      f"(clip = {duration_s:.1f}s)")
print(f"\n  First 5 events:")
print(df_events[['label','start','end','azimuth_deg','dist_m']].head().to_string(index=False))

# ── Build ambient windows (gaps ≥ 1s with no directional event) ─────────────
def build_ambient_windows(events_df, total_duration, min_gap_s=1.0):
    """Return list of (start, end) windows with zero directional activity."""
    busy = np.zeros(int(total_duration * SR), dtype=bool)
    for _, e in events_df.iterrows():
        s = int(e['start'] * SR)
        en = min(int(e['end']   * SR), len(busy) - 1)
        busy[s:en] = True
    # Find contiguous False runs
    windows = []
    in_gap, gap_start = False, 0
    for i, b in enumerate(busy):
        if not b and not in_gap:
            in_gap, gap_start = True, i
        elif b and in_gap:
            gap_end = i
            if (gap_end - gap_start) / SR >= min_gap_s:
                windows.append((gap_start / SR, gap_end / SR))
            in_gap = False
    if in_gap:
        gap_end = len(busy)
        if (gap_end - gap_start) / SR >= min_gap_s:
            windows.append((gap_start / SR, gap_end / SR))
    return windows

ambient_windows = build_ambient_windows(df_events, duration_s, min_gap_s=1.0)
ambient_total = sum(e - s for s, e in ambient_windows)
event_total   = (df_events['end'] - df_events['start']).sum()
print(f"\n  Ambient windows: {len(ambient_windows)}  "
      f"(total {ambient_total:.1f}s)")
print(f"  Event  windows: {len(df_events)}  (total {event_total:.1f}s)")

## 3. Compute Inter-Channel Phase Differences (IPD)

For each STFT frame and each mic pair $(i,j)$ the **Inter-channel Phase Difference** is:

$$\phi_{ij}(f, t) = \angle \bigl( X_i(f,t) \cdot X_j^*(f,t) \bigr)$$

For a directional source at azimuth $\theta$, the expected phase for mic pair with baseline vector $\mathbf{d}_{ij}$ is:

$$\phi_{ij}^{\text{dir}}(f) = 2\pi f \cdot \frac{\mathbf{d}_{ij} \cdot \hat{n}(\theta)}{c}$$

This is **linear in frequency** with slope = TDOA.  
For diffuse/ambient sound, phase is random → no linear structure.

We also compute the **GCC-PHAT** cross-correlation and track how *sharp* vs *flat* its peak is.

In [ ]:
window = scipy_signal.windows.hann(FRAME_SIZE)
freqs  = np.fft.rfftfreq(FRAME_SIZE, d=1.0/SR)   # Hz per bin

def stft_mics(mic_audio_chunk):
    """Compute STFT for all 4 mics. Returns (4, N_FREQS, n_frames) complex array."""
    n_ch, n_samp = mic_audio_chunk.shape
    n_frames = (n_samp - FRAME_SIZE) // HOP_SIZE + 1
    out = np.zeros((n_ch, N_FREQS, n_frames), dtype=complex)
    for t in range(n_frames):
        s = t * HOP_SIZE
        frame = mic_audio_chunk[:, s:s+FRAME_SIZE] * window
        out[:, :, t] = np.fft.rfft(frame, n=FRAME_SIZE, axis=1)
    return out

def compute_ipd(stft_all):
    """
    Compute IPD for all mic pairs.
    stft_all: (4, N_FREQS, T) complex
    Returns: dict pair_label -> (N_FREQS, T) phase array in radians
    """
    ipd = {}
    for (i, j), lbl in zip(MIC_PAIRS, PAIR_LABELS):
        cross = stft_all[i] * np.conj(stft_all[j])   # (N_FREQS, T)
        ipd[lbl] = np.angle(cross)
    return ipd

def gcc_phat(xi, xj):
    """GCC-PHAT between two mic spectra. Returns (N_FREQS,) real array (freq domain PHAT weights)."""
    cross = xi * np.conj(xj)
    denom = np.abs(cross) + 1e-12
    return cross / denom   # unit-magnitude cross-spectrum

def phase_slope_r2(phase_vec, freqs_subset, max_tdoa_s=None):
    """
    Fit a line φ = 2π·f·τ (zero-intercept) to the phase vs frequency curve.
    Returns (R², fitted_tdoa_s).
    Only use frequencies up to aliasing limit (c/2d) for the relevant pair.
    """
    if max_tdoa_s is None:
        max_tdoa_s = 0.064 * np.sqrt(2) / SPEED_OF_SOUND  # diagonal pair
    # Build design matrix: X = 2π·f
    X = (2 * np.pi * freqs_subset).reshape(-1, 1)
    y = phase_vec
    # Unwrap phase
    y_uw = np.unwrap(y)
    # Least-squares fit  y = τ·X  (zero intercept)
    tau = np.dot(X.ravel(), y_uw) / (np.dot(X.ravel(), X.ravel()) + 1e-30)
    y_hat = tau * X.ravel()
    ss_res = np.sum((y_uw - y_hat)**2)
    ss_tot = np.sum((y_uw - np.mean(y_uw))**2) + 1e-30
    r2 = max(0.0, 1.0 - ss_res / ss_tot)
    return r2, float(tau)

print("DSP helpers defined.")
print(f"  freqs range: {freqs[1]:.1f} – {freqs[-1]:.0f} Hz")
print(f"  Aliasing limit (32mm pair): {SPEED_OF_SOUND/(2*0.064):.0f} Hz  "
      f"(bin {int(SPEED_OF_SOUND/(2*0.064) / (SR/FRAME_SIZE))})")

In [ ]:
# ── Extract representative directional segments ──────────────────────────────
# Pick 40 random events (capped at min 0.5s duration) + 40 ambient windows
rng = np.random.default_rng(42)

def extract_segment(mic_audio, start_s, end_s, max_frames=60):
    """Extract mic audio segment and compute STFT."""
    s  = int(start_s * SR)
    e  = min(int(end_s   * SR), mic_audio.shape[1])
    chunk = mic_audio[:, s:e]
    if chunk.shape[1] < FRAME_SIZE:
        return None
    return stft_mics(chunk)[:, :, :max_frames]  # cap at max_frames

# Filter events long enough
valid_events = df_events[(df_events['end'] - df_events['start']) >= 0.5].copy()
chosen_events = valid_events.sample(min(50, len(valid_events)), random_state=42)

# Ambient: pick 50 random 1-second clips from ambient windows
ambient_clips = []
for (ws, we) in ambient_windows:
    n_clips = max(1, int((we - ws) / 1.5))
    starts = rng.uniform(ws, max(ws, we - 1.0), size=n_clips)
    for st in starts:
        ambient_clips.append((st, st + 1.0))
rng.shuffle(ambient_clips)
ambient_clips = ambient_clips[:50]

print(f"Directional segments to analyse : {len(chosen_events)}")
print(f"Ambient    segments to analyse  : {len(ambient_clips)}")

# ── Compute per-segment mean IPD and phase-slope R² across all pairs ─────────
def analyse_segment(stft_all):
    """
    For a segment STFT (4, N_FREQS, T):
    Returns dict with per-pair mean circular std of IPD, mean phase-slope R².
    Also returns per-pair mean MSC.
    """
    if stft_all is None or stft_all.shape[2] < 2:
        return None
    results = {}
    alias_bin = int(SPEED_OF_SOUND / (2 * 0.064) / (SR / FRAME_SIZE))
    freq_mask = (freqs >= 200) & (freqs <= alias_bin * SR / FRAME_SIZE)

    for (i, j), lbl in zip(MIC_PAIRS, PAIR_LABELS):
        xi = stft_all[i]   # (N_FREQS, T)
        xj = stft_all[j]

        # IPD across all frames
        cross = xi * np.conj(xj)                   # (N_FREQS, T)
        phi   = np.angle(cross)                     # (N_FREQS, T)

        # Mean circular std across time (per freq bin), then average over valid freqs
        circ_std_per_freq = circstd(phi[freq_mask, :], axis=1, nan_policy='omit')
        mean_circ_std = float(np.nanmean(circ_std_per_freq))

        # Phase-slope R² on mean (time-averaged) phase
        phi_mean = circmean(phi[freq_mask, :], axis=1, nan_policy='omit')
        r2, tau  = phase_slope_r2(phi_mean, freqs[freq_mask])

        # MSC (Magnitude Squared Coherence) — average over valid freqs
        Pxy = np.mean(cross[freq_mask, :], axis=1)           # averaged cross-PSD
        Pxx = np.mean(np.abs(xi[freq_mask, :])**2, axis=1)
        Pyy = np.mean(np.abs(xj[freq_mask, :])**2, axis=1)
        msc = np.abs(Pxy)**2 / (Pxx * Pyy + 1e-30)
        mean_msc = float(np.nanmean(msc))

        results[lbl] = {
            'circ_std': mean_circ_std,
            'r2'      : r2,
            'tau_ms'  : tau * 1000,
            'msc'     : mean_msc,
        }
    return results

print("Computing features for directional segments …")
dir_features = []
for _, ev in chosen_events.iterrows():
    stft = extract_segment(mic_audio, ev['start'], ev['end'])
    res  = analyse_segment(stft)
    if res:
        row = {'label': ev['label'], 'azimuth': ev['azimuth_deg'],
               'dist_m': ev['dist_m'], 'class': 'directional'}
        for lbl, v in res.items():
            for k, val in v.items():
                row[f'{lbl}_{k}'] = val
        dir_features.append(row)

print("Computing features for ambient segments …")
amb_features = []
for (ws, we) in ambient_clips:
    stft = extract_segment(mic_audio, ws, we)
    res  = analyse_segment(stft)
    if res:
        row = {'label': 'ambient', 'azimuth': np.nan,
               'dist_m': np.nan, 'class': 'ambient'}
        for lbl, v in res.items():
            for k, val in v.items():
                row[f'{lbl}_{k}'] = val
        amb_features.append(row)

df_dir = pd.DataFrame(dir_features)
df_amb = pd.DataFrame(amb_features)
df_all = pd.concat([df_dir, df_amb], ignore_index=True)

print(f"\nDirectional feature rows : {len(df_dir)}")
print(f"Ambient    feature rows  : {len(df_amb)}")

## 4. Visualize Phase Difference Distributions: Directional vs Ambient

Side-by-side IPD spectrograms and histograms for a representative directional event vs an ambient window.
We also plot the **time-averaged IPD vs frequency** curve — directional sources should show a clean linear ramp, ambient should be flat or noisy near zero.

In [ ]:
# ── Pick a concrete directional event (elephant — loud, distinctive) ─────────
ex_event = chosen_events[chosen_events['label'] == 'Elephant'].iloc[0]
stft_dir = extract_segment(mic_audio, ex_event['start'], ex_event['end'])

# Pick an ambient clip
stft_amb = extract_segment(mic_audio, *ambient_clips[3])

pair_plot = 'L-R'   # Left–Right pair (most sensitive to left/right azimuths)
pi_idx = PAIR_LABELS.index(pair_plot)
mic_i, mic_j = MIC_PAIRS[pi_idx]

def get_ipd_spec(stft_all, mi, mj):
    cross = stft_all[mi] * np.conj(stft_all[mj])
    return np.angle(cross)   # (N_FREQS, T)

ipd_dir = get_ipd_spec(stft_dir, mic_i, mic_j)
ipd_amb = get_ipd_spec(stft_amb, mic_i, mic_j)

alias_bin = int(SPEED_OF_SOUND / (2 * 0.064) / (SR / FRAME_SIZE))
freq_mask = (freqs >= 100) & (np.arange(N_FREQS) <= alias_bin)

fig, axes = plt.subplots(3, 2, figsize=(14, 12))
fig.suptitle('IPD Analysis — Directional (Elephant) vs Ambient (Forest background)',
             fontsize=13, fontweight='bold')

# ── Row 1: IPD spectrograms ───────────────────────────────────────────────────
for col, (ipd, title, stft_) in enumerate([
        (ipd_dir, f'DIRECTIONAL — Elephant @ {ex_event["azimuth_deg"]:.0f}°', stft_dir),
        (ipd_amb, 'AMBIENT — Forest background',                                stft_amb)]):
    ax = axes[0, col]
    T_axis = np.arange(ipd.shape[1]) * HOP_SIZE / SR
    im = ax.pcolormesh(T_axis, freqs[:alias_bin], ipd[:alias_bin, :],
                       cmap='twilight', vmin=-np.pi, vmax=np.pi, shading='auto')
    ax.set_ylabel('Frequency (Hz)')
    ax.set_xlabel('Time (s)')
    ax.set_title(f'IPD spectrogram  ({pair_plot})\n{title}', fontsize=10)
    plt.colorbar(im, ax=ax, label='Phase (rad)')

# ── Row 2: Time-averaged IPD vs frequency (linear fit) ───────────────────────
for col, (ipd, label, color) in enumerate([
        (ipd_dir, 'Directional', 'steelblue'),
        (ipd_amb, 'Ambient',     'tomato')]):
    ax = axes[1, col]
    phi_mean = circmean(ipd[freq_mask, :], axis=1, nan_policy='omit')
    phi_std  = circstd( ipd[freq_mask, :], axis=1, nan_policy='omit')
    fx = freqs[freq_mask]
    r2, tau = phase_slope_r2(phi_mean, fx)
    phi_fit = 2 * np.pi * fx * tau

    ax.fill_between(fx, phi_mean - phi_std, phi_mean + phi_std,
                    alpha=0.25, color=color, label='±1 σ')
    ax.plot(fx, np.unwrap(phi_mean), color=color, lw=1.5, label='mean IPD')
    ax.plot(fx, np.unwrap(phi_fit),  color='k',   lw=1.2, ls='--',
            label=f'linear fit  τ={tau*1000:.3f}ms  R²={r2:.3f}')
    ax.set_xlabel('Frequency (Hz)')
    ax.set_ylabel('Phase (rad, unwrapped)')
    ax.set_title(f'Time-averaged IPD vs freq — {label}', fontsize=10)
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

# ── Row 3: IPD histograms across all freq-time bins ──────────────────────────
for col, (ipd, label, color) in enumerate([
        (ipd_dir, 'Directional', 'steelblue'),
        (ipd_amb, 'Ambient',     'tomato')]):
    ax = axes[2, col]
    phi_flat = ipd[freq_mask, :].ravel()
    ax.hist(phi_flat, bins=72, range=(-np.pi, np.pi), color=color,
            edgecolor='white', linewidth=0.3, density=True)
    circ_s = circstd(phi_flat, nan_policy='omit')
    ax.axvline(0, color='k', ls='--', lw=1.2)
    ax.set_xlabel('IPD (rad)')
    ax.set_ylabel('Density')
    ax.set_title(f'IPD histogram — {label}\nCircular std = {circ_s:.3f} rad', fontsize=10)
    ax.set_xlim(-np.pi, np.pi)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('/home/azureuser/simulator/outputs/analysis/phase_ipd_comparison.png',
            dpi=150, bbox_inches='tight')
plt.show()
print("Saved → outputs/analysis/phase_ipd_comparison.png")

## 5. Statistical Analysis of Phase Coherence

Compute per-segment **Magnitude Squared Coherence (MSC)** and compare it to the
**diffuse-field spatial coherence model**:

$$C_{\text{diffuse}}(f, d) = \left| \text{sinc}\!\left(\frac{2fd}{c}\right) \right|^2 = \left| \frac{\sin(2\pi f d / c)}{2\pi f d / c} \right|^2$$

A diffuse ambient field *already* has predictable coherence — the sinc shape — even though it's omnidirectional.  
A directional source produces **higher** MSC than the diffuse model (more coherent than predicted by diffuse field).

**Key insight:** If measured MSC ≫ sinc model → likely directional.  
If measured MSC ≈ sinc model → ambient/diffuse, suppress regardless of amplitude.

In [ ]:
alias_bin = int(SPEED_OF_SOUND / (2 * 0.064) / (SR / FRAME_SIZE))  # ensure defined here

def diffuse_field_msc(freqs_hz, d_m):
    """Theoretical MSC for spatially incoherent (diffuse) field between 2 omni mics at distance d."""
    x = 2 * np.pi * freqs_hz * d_m / SPEED_OF_SOUND
    sinc_val = np.sinc(x / np.pi)   # np.sinc(x/π) = sin(x)/x
    return sinc_val**2

def compute_msc_spectrum(stft_all):
    """
    Welch-style MSC: average cross- and auto-spectra over all frames, then compute MSC.
    Returns dict pair→(N_FREQS,) MSC array  ∈ [0, 1].
    """
    msc_dict = {}
    for (i, j), lbl in zip(MIC_PAIRS, PAIR_LABELS):
        xi = stft_all[i]   # (N_FREQS, T)
        xj = stft_all[j]
        Pxy = np.mean(xi * np.conj(xj), axis=1)
        Pxx = np.mean(np.abs(xi)**2, axis=1)
        Pyy = np.mean(np.abs(xj)**2, axis=1)
        msc = np.abs(Pxy)**2 / (Pxx * Pyy + 1e-30)
        msc_dict[lbl] = np.clip(msc, 0, 1)
    return msc_dict

# ── Compute MSC for all three tiers ──────────────────────────────────────────
msc_dir    = compute_msc_spectrum(stft_dir)
msc_amb    = compute_msc_spectrum(stft_amb)
msc_synth  = compute_msc_spectrum(stft_check)   # synthetic diffuse from cell above

pair_distances = {}
for (i, j), lbl in zip(MIC_PAIRS, PAIR_LABELS):
    pair_distances[lbl] = np.linalg.norm(MIC_POSITIONS[j] - MIC_POSITIONS[i])

# ── Plot: all three tiers + theoretical diffuse model ────────────────────────
fig, axes = plt.subplots(2, 3, figsize=(15, 9))
fig.suptitle(
    'Magnitude Squared Coherence — Three-Tier Comparison\n'
    'Directional (blue) | Real ambient (red) | Synthetic diffuse (orange) | Sinc² theory (dashed)',
    fontsize=12, fontweight='bold')

for ax, lbl in zip(axes.ravel(), PAIR_LABELS):
    d       = pair_distances[lbl]
    theory  = diffuse_field_msc(freqs[:alias_bin], d)

    ax.plot(freqs[:alias_bin], msc_dir  [lbl][:alias_bin],
            color='steelblue',  lw=2.0, label='Directional event')
    ax.plot(freqs[:alias_bin], msc_amb  [lbl][:alias_bin],
            color='tomato',     lw=1.6, label='Real ambient (forest gaps)')
    ax.plot(freqs[:alias_bin], msc_synth[lbl][:alias_bin],
            color='darkorange', lw=1.6, label='Synthetic isotropic diffuse')
    ax.plot(freqs[:alias_bin], theory,
            color='black', lw=1.2, ls='--', label='Sinc² theory (ideal diffuse)')

    # Highlight the useful separation band
    ax.fill_between(freqs[:alias_bin],
                    msc_dir[lbl][:alias_bin], msc_amb[lbl][:alias_bin],
                    where=msc_dir[lbl][:alias_bin] > msc_amb[lbl][:alias_bin],
                    alpha=0.12, color='steelblue', label='Dir−Amb gap (exploitable)')

    ax.set_title(f'Pair {lbl}  (d={d*100:.1f} cm)', fontsize=10)
    ax.set_xlabel('Frequency (Hz)')
    ax.set_ylabel('MSC')
    ax.set_ylim(0, 1.05)
    ax.legend(fontsize=7)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('/home/azureuser/simulator/outputs/analysis/phase_msc_three_tiers.png',
            dpi=150, bbox_inches='tight')
plt.show()

# ── Key diagnostic table ──────────────────────────────────────────────────────
print("\nMean MSC (200 Hz – aliasing limit) per tier and pair:\n")
print(f"{'Pair':<6} {'Directional':>13} {'Real ambient':>14} {'Synth diffuse':>14} "
      f"{'Sinc² theory':>14}  {'Dir/Amb ratio':>14}")
print("─" * 80)
for lbl in PAIR_LABELS:
    d      = pair_distances[lbl]
    theory = diffuse_field_msc(freqs[:alias_bin], d)
    freq_m = (freqs[:alias_bin] >= 200)
    m_dir   = float(np.mean(msc_dir  [lbl][:alias_bin][freq_m]))
    m_amb   = float(np.mean(msc_amb  [lbl][:alias_bin][freq_m]))
    m_syn   = float(np.mean(msc_synth[lbl][:alias_bin][freq_m]))
    m_th    = float(np.mean(theory[freq_m]))
    ratio   = m_dir / (m_amb + 1e-6)
    print(f"{lbl:<6} {m_dir:>13.3f} {m_amb:>14.3f} {m_syn:>14.3f} {m_th:>14.3f}  {ratio:>14.1f}×")

print("\n📌 Key interpretation:")
print("  Synth diffuse ≈ Sinc² theory  → generator validated")
print("  Real ambient > Synth diffuse   → forest background is NOT truly diffuse (expected!)")
print("  Directional >> Real ambient    → there IS exploitable separation")

In [ ]:
# ── Population statistics across 50 directional + 50 real-ambient segments ────
# Also compute features for the synthetic diffuse (single segment, repeated for reference)

# First compute features for 10 x 1s synthetic diffuse chunks for a distribution
print("Computing features for synthetic diffuse segments …")
synth_features = []
for chunk_idx in range(30):
    sd_chunk = generate_synthetic_diffuse(1.0, n_sources=120, seed=100 + chunk_idx)
    stft_sd_c = stft_mics(sd_chunk)
    res = analyse_segment(stft_sd_c)
    if res:
        row = {'label': 'synth_diffuse', 'azimuth': np.nan,
               'dist_m': np.nan, 'class': 'synth_diffuse'}
        for lbl, v in res.items():
            for k, val in v.items():
                row[f'{lbl}_{k}'] = val
        synth_features.append(row)

df_synth = pd.DataFrame(synth_features)
print(f"  Synthetic diffuse feature rows: {len(df_synth)}")

# ── Plot distributions — all three tiers ──────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle(
    'Population distributions — Directional vs Real Ambient vs Synthetic Diffuse\n'
    '(Real ambient is NOT truly diffuse — it sits between synth diffuse and directional)',
    fontsize=11, fontweight='bold')

r2_cols  = [f'{lbl}_r2'       for lbl in PAIR_LABELS]
msc_cols = [f'{lbl}_msc'      for lbl in PAIR_LABELS]
std_cols = [f'{lbl}_circ_std' for lbl in PAIR_LABELS]

tiers = [
    (df_dir,   'steelblue',  'Directional events'),
    (df_amb,   'tomato',     'Real ambient (forest gaps)'),
    (df_synth, 'darkorange', 'Synthetic isotropic diffuse'),
]

for ax, cols, xlabel in zip(axes,
    [r2_cols, msc_cols, std_cols],
    ['Phase-Slope R² (mean across pairs)',
     'Magnitude Squared Coherence (mean)',
     'Circular Std of IPD (rad, mean)']):
    for df, color, label in tiers:
        vals = df[cols].mean(axis=1).dropna()
        ax.hist(vals, bins=20, color=color, alpha=0.55, edgecolor='white',
                density=True, label=f'{label}\n  μ={vals.mean():.3f}  σ={vals.std():.3f}')
    ax.set_xlabel(xlabel)
    ax.set_ylabel('Density')
    ax.legend(fontsize=7.5)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('/home/azureuser/simulator/outputs/analysis/phase_population_three_tiers.png',
            dpi=150, bbox_inches='tight')
plt.show()

# ── Summary table ──────────────────────────────────────────────────────────────
print("\nSegment-level statistics (mean over all 6 mic pairs):\n")
print(f"{'Metric':<12} {'Directional':>16} {'Real ambient':>16} {'Synth diffuse':>16}")
print("─" * 64)
for metric, cols in [('R²', r2_cols), ('MSC', msc_cols), ('CircStd', std_cols)]:
    d_v = df_dir  [cols].mean(axis=1).dropna()
    a_v = df_amb  [cols].mean(axis=1).dropna()
    s_v = df_synth[cols].mean(axis=1).dropna()
    print(f"{metric:<12} {d_v.mean():>8.3f}±{d_v.std():.3f}  "
          f"{a_v.mean():>8.3f}±{a_v.std():.3f}  "
          f"{s_v.mean():>8.3f}±{s_v.std():.3f}")

print()
print("📌 What to look for:")
print("  • Synth diffuse should have the LOWEST R², lowest MSC, highest CircStd")
print("  • Real ambient should be in the MIDDLE (it has directional forest content)")
print("  • Directional events should have the HIGHEST R², highest MSC, lowest CircStd")
print("  • If Real ambient overlaps heavily with Directional → harder discrimination problem")

## 6. Design Phase-Based Ambient Sound Rejection Filter

### What we're actually separating

The three-tier analysis tells us the real picture:

| Tier | MSC profile | R² | Interpretation |
|---|---|---|---|
| **Synthetic diffuse** | Matches sinc² exactly | ~0 | Mathematical lower bound — perfectly diffuse |
| **Real ambient (forest)** | Above sinc², some structure | Low-medium | Distributed directional sources, aggregate partially smears → sits above sync² but below a strong event |
| **Directional event** | Well above sinc², spectrally consistent | High | One dominant TDOA → MSC locks on |

**The discriminator threshold** should be placed between the real-ambient distribution and the directional-event distribution — **not** at the theoretical diffuse floor. The synthetic diffuse just validates that our metrics work as expected.

**Directionality Score** $D$ per frame:

$$D = w_1 \cdot \bar{R}^2 + w_2 \cdot \overline{\Delta\text{MSC}} + w_3 \cdot (1 - \bar{\sigma}_\phi / \pi)$$

where $\overline{\Delta\text{MSC}} = \text{mean}(\text{MSC}_\text{measured} - \text{sinc}^2_\text{theory})$ clipped to $[0,1]$.

A frame is **suppressed** (treated as ambient) if $D < \tau$.  
Critically, suppression is **amplitude-blind** — a very loud forest chorus at a hotspot azimuth gets suppressed just like a quiet one, because the discriminator is spatial coherence, not energy.

In [ ]:
def directionality_score(stft_all, w1=0.4, w2=0.4, w3=0.2):
    """
    Compute per-frame Directionality Score D ∈ [0, 1].
    
    Higher D = more likely to be a directional source.
    Lower  D = more likely to be diffuse ambient sound.
    
    Parameters
    ----------
    stft_all : (4, N_FREQS, T) complex
    w1, w2, w3 : weights for R², MSC_excess, 1-circ_std/π
    
    Returns
    -------
    scores : (T,) float array
    """
    T = stft_all.shape[2]
    alias_bin = int(SPEED_OF_SOUND / (2 * 0.064) / (SR / FRAME_SIZE))
    freq_mask_idx = np.where((freqs >= 200) & (np.arange(N_FREQS) <= alias_bin))[0]
    fq = freqs[freq_mask_idx]

    r2_sum = np.zeros(T)
    msc_exc_sum = np.zeros(T)
    cstd_sum = np.zeros(T)
    n_pairs = len(MIC_PAIRS)

    for (i, j), lbl in zip(MIC_PAIRS, PAIR_LABELS):
        d = pair_distances[lbl]
        xi = stft_all[i][freq_mask_idx, :]   # (F, T)
        xj = stft_all[j][freq_mask_idx, :]

        cross = xi * np.conj(xj)   # (F, T)
        phi   = np.angle(cross)    # (F, T)

        # Per-frame phase-slope R²
        X = (2 * np.pi * fq)       # (F,)
        for t in range(T):
            phi_t = np.unwrap(phi[:, t])
            tau_t = np.dot(X, phi_t) / (np.dot(X, X) + 1e-30)
            phi_hat = tau_t * X
            ss_res = np.sum((phi_t - phi_hat)**2)
            ss_tot = np.sum((phi_t - np.mean(phi_t))**2) + 1e-30
            r2_sum[t] += max(0.0, 1.0 - ss_res / ss_tot)

        # Per-frame MSC excess over diffuse model
        diffuse = diffuse_field_msc(fq, d)     # (F,)
        for t in range(T):
            Pxy = cross[:, t]                  # already instantaneous
            Pxx = np.abs(xi[:, t])**2 + 1e-30
            Pyy = np.abs(xj[:, t])**2 + 1e-30
            msc_t = np.abs(Pxy)**2 / (Pxx * Pyy)
            msc_exc_sum[t] += float(np.mean(np.clip(msc_t - diffuse, 0, 1)))

        # Per-frame circular std
        cstd_sum += circstd(phi, axis=0, nan_policy='omit')

    r2_mean    = r2_sum    / n_pairs
    msc_mean   = msc_exc_sum / n_pairs
    cstd_mean  = cstd_sum  / n_pairs

    # Normalize MSC excess to [0,1]: max theoretical excess ≈ 1 - 0 = 1
    msc_norm = np.clip(msc_mean / 0.5, 0, 1)   # 0.5 is a rough upper practical value
    cstd_norm = np.clip(1.0 - cstd_mean / np.pi, 0, 1)

    D = w1 * r2_mean + w2 * msc_norm + w3 * cstd_norm
    return np.clip(D, 0, 1)

# ── Compute scores for example segments ─────────────────────────────────────
scores_dir = directionality_score(stft_dir)
scores_amb = directionality_score(stft_amb)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Per-Frame Directionality Score D — Directional vs Ambient',
             fontsize=12, fontweight='bold')

t_dir = np.arange(len(scores_dir)) * HOP_SIZE / SR
t_amb = np.arange(len(scores_amb)) * HOP_SIZE / SR

for ax, t, scores, color, title in [
        (axes[0], t_dir, scores_dir, 'steelblue',
         f'Directional — Elephant @ {ex_event["azimuth_deg"]:.0f}°'),
        (axes[1], t_amb, scores_amb, 'tomato',
         'Ambient — Forest background')]:
    ax.plot(t, scores, color=color, lw=1.5)
    ax.axhline(scores.mean(), color=color, ls='--', lw=1.2,
               label=f'Mean D = {scores.mean():.3f}')
    for thresh in [0.3, 0.5]:
        ax.axhline(thresh, color='gray', ls=':', lw=0.8)
        ax.text(t[-1]*0.02, thresh+0.01, f'τ={thresh}', fontsize=8, color='gray')
    ax.set_ylim(0, 1.1)
    ax.set_xlabel('Time (s)')
    ax.set_ylabel('Directionality Score D')
    ax.set_title(title, fontsize=10)
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('/home/azureuser/simulator/outputs/analysis/phase_directionality_score.png',
            dpi=150, bbox_inches='tight')
plt.show()

print(f"\nDirectional segment:  mean D = {scores_dir.mean():.3f}  "
      f"median D = {np.median(scores_dir):.3f}")
print(f"Ambient    segment:  mean D = {scores_amb.mean():.3f}  "
      f"median D = {np.median(scores_amb):.3f}")

## 7. Evaluate Filter Against Known Events (Recall & False Positive Rate)

Run the Directionality Score across **all 50 directional + 50 ambient** segments.  
A segment is classified as "directional" if its **mean D ≥ threshold τ**.

- **True Positive (TP):** directional segment correctly flagged as directional
- **False Negative (FN):** directional segment wrongly suppressed (= missed event)
- **True Negative (TN):** ambient segment correctly suppressed
- **False Positive (FP):** ambient segment wrongly passed through (= phantom track)

In [ ]:
print("Computing Directionality Score for all segments … (may take ~30s)")

dir_mean_scores = []
for _, ev in chosen_events.iterrows():
    stft = extract_segment(mic_audio, ev['start'], ev['end'])
    if stft is not None and stft.shape[2] >= 2:
        s = directionality_score(stft)
        dir_mean_scores.append({'class': 'directional', 'label': ev['label'],
                                 'mean_D': float(s.mean()), 'median_D': float(np.median(s))})
    else:
        dir_mean_scores.append({'class': 'directional', 'label': ev['label'],
                                 'mean_D': np.nan, 'median_D': np.nan})

amb_mean_scores = []
for (ws, we) in ambient_clips:
    stft = extract_segment(mic_audio, ws, we)
    if stft is not None and stft.shape[2] >= 2:
        s = directionality_score(stft)
        amb_mean_scores.append({'class': 'ambient', 'label': 'ambient',
                                 'mean_D': float(s.mean()), 'median_D': float(np.median(s))})
    else:
        amb_mean_scores.append({'class': 'ambient', 'label': 'ambient',
                                 'mean_D': np.nan, 'median_D': np.nan})

df_scores = pd.DataFrame(dir_mean_scores + amb_mean_scores).dropna()

# ── Box-plot by class & label ─────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle('Directionality Score D by Class and Label', fontsize=12, fontweight='bold')

# By class
classes = df_scores['class'].unique()
data_by_class = [df_scores[df_scores['class'] == c]['mean_D'].values for c in classes]
colors_cls = ['steelblue', 'tomato']
bp = axes[0].boxplot(data_by_class, labels=classes, patch_artist=True,
                     medianprops=dict(color='black', lw=2))
for patch, color in zip(bp['boxes'], colors_cls):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)
axes[0].set_ylabel('Mean D per segment')
axes[0].set_title('Directional vs Ambient')
axes[0].grid(True, alpha=0.3)
for val in [0.3, 0.5]:
    axes[0].axhline(val, color='gray', ls=':', lw=1)
    axes[0].text(0.52, val+0.01, f'τ={val}', fontsize=8, color='gray')

# By label (directional only)
labels_dir = df_scores[df_scores['class'] == 'directional']['label'].unique()
data_by_lbl = [df_scores[(df_scores['class']=='directional') & 
                          (df_scores['label']==lbl)]['mean_D'].values
               for lbl in labels_dir]
axes[1].boxplot(data_by_lbl, labels=labels_dir, patch_artist=True,
                medianprops=dict(color='black', lw=2))
axes[1].set_ylabel('Mean D per segment')
axes[1].set_title('Directional sources by species')
axes[1].tick_params(axis='x', rotation=30)
axes[1].grid(True, alpha=0.3)
for val in [0.3, 0.5]:
    axes[1].axhline(val, color='gray', ls=':', lw=1)

plt.tight_layout()
plt.savefig('/home/azureuser/simulator/outputs/analysis/phase_score_by_class.png',
            dpi=150, bbox_inches='tight')
plt.show()

# ── Recall / FPR at a fixed threshold ────────────────────────────────────────
THRESHOLD = 0.35   # tunable — will sweep in next section
dir_scores_arr = df_scores[df_scores['class']=='directional']['mean_D'].values
amb_scores_arr = df_scores[df_scores['class']=='ambient'   ]['mean_D'].values

tp = np.sum(dir_scores_arr >= THRESHOLD)
fn = np.sum(dir_scores_arr <  THRESHOLD)
tn = np.sum(amb_scores_arr <  THRESHOLD)
fp = np.sum(amb_scores_arr >= THRESHOLD)

recall  = tp / (tp + fn + 1e-9)
fpr     = fp / (fp + tn + 1e-9)
precision = tp / (tp + fp + 1e-9)

print(f"\nAt threshold τ = {THRESHOLD}:")
print(f"  Recall    (sensitivity) = {recall:.3f}   ({tp} TP, {fn} FN)")
print(f"  FPR (ambient pass-thru) = {fpr:.3f}   ({fp} FP, {tn} TN)")
print(f"  Precision               = {precision:.3f}")
print(f"\nBaseline ODAS (from guide): recall ~{0.61:.2f}, no ambient pre-filter")

## 8. Threshold Tuning and ROC Curve Analysis

Sweep the coherence rejection threshold τ from 0 → 1 and compute the full ROC curve.  
The **optimal operating point** balances:
- High recall (don't miss real events — currently 61% baseline)
- Low FPR (don't pass ambient as directional — currently ~0.63 FP/s)

We also look for per-label recall at the chosen threshold — hard-to-detect species like `bear`, `frog`, `drone_binary` are most at risk of being suppressed.

In [ ]:
thresholds = np.linspace(0.0, 1.0, 200)

recalls, fprs, precisions, f1s = [], [], [], []
for tau in thresholds:
    tp_ = np.sum(dir_scores_arr >= tau)
    fn_ = np.sum(dir_scores_arr <  tau)
    tn_ = np.sum(amb_scores_arr <  tau)
    fp_ = np.sum(amb_scores_arr >= tau)
    rec  = tp_ / (tp_ + fn_ + 1e-9)
    fpr_ = fp_ / (fp_ + tn_ + 1e-9)
    prec = tp_ / (tp_ + fp_ + 1e-9)
    f1_  = 2 * prec * rec / (prec + rec + 1e-9)
    recalls.append(rec); fprs.append(fpr_)
    precisions.append(prec); f1s.append(f1_)

recalls    = np.array(recalls)
fprs       = np.array(fprs)
f1s        = np.array(f1s)
precisions = np.array(precisions)

best_idx   = np.argmax(f1s)
best_tau   = thresholds[best_idx]
best_f1    = f1s[best_idx]
best_rec   = recalls[best_idx]
best_fpr   = fprs[best_idx]

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('ROC Analysis — Phase-Based Directionality Score', fontsize=12, fontweight='bold')

# ── ROC curve ────────────────────────────────────────────────────────────────
axes[0].plot(fprs, recalls, color='navy', lw=2)
axes[0].scatter([best_fpr], [best_rec], color='red', s=80, zorder=5,
                label=f'Best F1 τ={best_tau:.2f}\nRecall={best_rec:.3f} FPR={best_fpr:.3f}')
axes[0].plot([0,1],[0,1],'k--',lw=0.8, label='Random')
axes[0].set_xlabel('False Positive Rate (ambient pass-thru)')
axes[0].set_ylabel('Recall (TP rate for directional)')
axes[0].set_title('ROC Curve')
axes[0].legend(fontsize=8)
axes[0].grid(True, alpha=0.3)
# Mark baseline ODAS recall
axes[0].axhline(0.61, color='orange', ls='--', lw=1, label='ODAS baseline recall')

# ── Recall + Precision vs threshold ──────────────────────────────────────────
axes[1].plot(thresholds, recalls,    color='steelblue', lw=2, label='Recall')
axes[1].plot(thresholds, precisions, color='green',     lw=2, label='Precision')
axes[1].plot(thresholds, f1s,        color='purple',    lw=2, label='F1')
axes[1].axvline(best_tau, color='red', ls='--', lw=1.5, label=f'Best τ={best_tau:.2f}')
axes[1].axhline(0.61, color='orange', ls=':', lw=1)
axes[1].set_xlabel('Threshold τ')
axes[1].set_ylabel('Score')
axes[1].set_title('Recall / Precision / F1 vs τ')
axes[1].legend(fontsize=8)
axes[1].grid(True, alpha=0.3)

# ── Per-label recall at best threshold ───────────────────────────────────────
dir_df_with_scores = df_scores[df_scores['class'] == 'directional'].copy()
label_recalls = {}
for lbl in dir_df_with_scores['label'].unique():
    vals = dir_df_with_scores[dir_df_with_scores['label']==lbl]['mean_D'].values
    label_recalls[lbl] = np.mean(vals >= best_tau) if len(vals) > 0 else 0.0

labels_sorted = sorted(label_recalls, key=lambda l: label_recalls[l])
recall_vals   = [label_recalls[l] for l in labels_sorted]
colors_bar    = ['tomato' if r < 0.7 else 'steelblue' for r in recall_vals]

axes[2].barh(labels_sorted, recall_vals, color=colors_bar, edgecolor='white')
axes[2].axvline(best_rec, color='navy', ls='--', lw=1.5, label=f'Overall recall={best_rec:.2f}')
axes[2].axvline(0.61, color='orange', ls=':', lw=1.2, label='ODAS baseline')
axes[2].set_xlabel('Recall at best τ')
axes[2].set_title(f'Per-label recall  (τ={best_tau:.2f})')
axes[2].set_xlim(0, 1.1)
axes[2].legend(fontsize=8)
axes[2].grid(True, alpha=0.3, axis='x')

plt.tight_layout()
plt.savefig('/home/azureuser/simulator/outputs/analysis/phase_roc_analysis.png',
            dpi=150, bbox_inches='tight')
plt.show()

print(f"\n{'='*60}")
print(f"  OPTIMAL THRESHOLD  τ* = {best_tau:.3f}")
print(f"{'='*60}")
print(f"  Overall recall    : {best_rec:.3f}   (baseline ODAS: 0.61)")
print(f"  Ambient rejection : {1-best_fpr:.3f}   (FPR: {best_fpr:.3f})")
print(f"  Precision         : {precisions[best_idx]:.3f}")
print(f"  F1                : {best_f1:.3f}")
print(f"\nPer-label recall at τ*={best_tau:.3f}:")
for lbl in sorted(label_recalls, key=lambda l: label_recalls[l]):
    flag = '⚠️ ' if label_recalls[lbl] < 0.65 else '✅ '
    print(f"  {flag}{lbl:<20} {label_recalls[lbl]:.3f}")

## 9. Findings & Recommendations

### What we confirmed (or refuted) empirically

Run the cells above and fill in the table with actual numbers:

| Metric | Directional (mean) | Ambient (mean) | Separation |
|--------|-------------------|----------------|-----------|
| Phase-slope R² | — | — | — |
| MSC (excess over diffuse) | — | — | — |
| Circular Std of IPD | — | — | — |
| Directionality Score D | — | — | — |

### Integration with ODAS pipeline

**Where to inject the Directionality Score:**

```
Raw audio frame (8ms hop)
       ↓
  Compute STFT (all 4 mics)
       ↓
  Compute D = directionality_score(stft)   ← NEW GATE
       ↓
  if D < τ*:
      suppress frame entirely (do NOT feed GCC-PHAT)
  else:
      continue → GCC-PHAT → SSL peak → SST tracking
```

This runs **before** GCC-PHAT, so ambient-born peaks never reach the SSL stage.  
The structural hotspot FPs (`-90°, 0°, -120°`) will be killed because they arise in ambient frames.

### Parameter recommendations (to be validated)

| Parameter | Current | Proposed with phase gate |
|-----------|---------|--------------------------|
| `Pnew` | 0.06 | Can raise to 0.15 (phase gate compensates) |
| `N_prob` | 6 | Can reduce to 3–4 (fewer ambient frames reaching SSL) |
| `theta_prob` | 0.65 | Keep or raise slightly |
| `gainMin` | 0.40 | Can lower to 0.30 (phase gate prevents ambient FPs) |
| **Phase gate τ*** | new | 0.30–0.40 (from ROC, see above) |

### Next steps

1. **Implement as a real-time Python pre-filter** in `odas_optimized.py` — add `PhaseGate` class that computes D per frame and skips `_compute_ssl()` when D < τ
2. **Cross-validate** with full 1220s scene (not just 50 segments)
3. **Tune weights w1, w2, w3** — try a grid search or simple Bayesian optimization
4. **Test on hard labels** — drone_binary and bear need special attention; verify their D scores are not systematically low (small distance, low amplitude → might have low R²)
5. **Port concept to C** if Python results are strong — the key operations are just cross-spectrum, FFT, and linear regression, all feasible in the ODAS C codebase